In [7]:
"""
Churn Prediction Model Training Pipeline
Dataset: BankChurners.csv
Output : churn_model.pkl, encoders.pkl, model_report.txt
"""

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report)
from xgboost import XGBClassifier

DATA_PATH  = r'D:\ITI\python visualization\Day02\project\files\BankChurners.csv'
MODEL_DIR  = r'D:\ITI\python visualization\Day02\project\files'

print('=' * 60)
print('CHURN PREDICTION MODEL TRAINING')
print('=' * 60)


ModuleNotFoundError: No module named 'xgboost'

# Assistant
This error occurs because the XGBoost library is not installed in your environment. The error message "ModuleNotFoundError: No module named 'xgboost'" indicates that Python cannot find the xgboost module that you're trying to import.

Would you like me to provide the corrected code?

# Assistant
This error occurs because the XGBoost library is not installed in your environment. The error message "ModuleNotFoundError: No module named 'xgboost'" indicates that Python cannot find the xgboost module that you're trying to import.

Would you like me to provide the corrected code?

# Assistant
This error occurs because the XGBoost library is not installed in your environment. The error message "ModuleNotFoundError: No module named 'xgboost'" indicates that Python cannot find the xgboost module that you're trying to import.

Would you like me to provide the corrected code?

# Assistant
This error occurs because the XGBoost library is not installed in your environment. The error message "ModuleNotFoundError: No module named 'xgboost'" indicates that Python cannot find the xgboost module that you're trying to import.

Would you like me to provide the corrected code?

# Assistant
This error occurs because the XGBoost library is not installed in your environment. The error message "ModuleNotFoundError: No module named 'xgboost'" indicates that Python cannot find the xgboost module that you're trying to import.

Would you like me to provide the corrected code?

# Assistant
This error occurs because the XGBoost library is not installed in your environment. The error message "ModuleNotFoundError: No module named 'xgboost'" indicates that Python cannot find the xgboost module that you're trying to import.

Would you like me to provide the corrected code?

# Assistant
This error occurs because the XGBoost library is not installed in your environment. The error message "ModuleNotFoundError: No module named 'xgboost'" indicates that Python cannot find the xgboost module that you're trying to import.

Would you like me to provide the corrected code?

# ──────────────────────────────────────────────────────────────
# STEP 1 — LOAD & CLEAN DATA
# ──────────────────────────────────────────────────────────────

In [6]:
print('\n[1/6] Loading data...')
df = pd.read_csv(DATA_PATH)
print(f'  Loaded {df.shape[0]} rows x {df.shape[1]} columns')

# Drop useless columns
df.drop(columns=['CLIENTNUM', df.columns[-1], df.columns[-2]], inplace=True)

# Encode target
df['Attrition_Flag'] = df['Attrition_Flag'].map({
    'Existing Customer': 0,
    'Attrited Customer': 1
})

# Encode Gender
df['Gender'] = df['Gender'].map({'M': 0, 'F': 1})

# Ordinal encode categorical columns (Unknown kept as a category)
encoders = {}

education_order = ['Unknown', 'Uneducated', 'High School', 'College',
                   'Graduate', 'Post-Graduate', 'Doctorate']
income_order    = ['Unknown', 'Less than $40K', '$40K - $60K',
                   '$60K - $80K', '$80K - $120K', '$120K +']
card_order      = ['Blue', 'Silver', 'Gold', 'Platinum']
marital_order   = ['Unknown', 'Single', 'Married', 'Divorced']

for col, order in [('Education_Level', education_order),
                   ('Income_Category', income_order),
                   ('Card_Category', card_order),
                   ('Marital_Status', marital_order)]:
    enc = OrdinalEncoder(categories=[order])
    df[col] = enc.fit_transform(df[[col]])
    encoders[col] = order

# Cap outliers
cap_cols = ['Credit_Limit', 'Avg_Open_To_Buy', 'Total_Trans_Amt',
            'Total_Amt_Chng_Q4_Q1', 'Total_Ct_Chng_Q4_Q1',
            'Contacts_Count_12_mon', 'Months_on_book']
for col in cap_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

print(f'  Final shape: {df.shape}')


[1/6] Loading data...


NameError: name 'DATA_PATH' is not defined

# ──────────────────────────────────────────────────────────────
# STEP 2 — SPLIT FEATURES / TARGET
# ──────────────────────────────────────────────────────────────

In [ ]:
print('\n[2/6] Splitting features / target...')
X = df.drop(columns=['Attrition_Flag'])
y = df['Attrition_Flag']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'  Train: {X_train.shape[0]} rows  |  Test: {X_test.shape[0]} rows')
print(f'  Churn rate (train): {y_train.mean()*100:.2f}%')
print(f'  Churn rate (test) : {y_test.mean()*100:.2f}%')

# Compute class imbalance ratio for XGBoost
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f'  scale_pos_weight  : {scale_pos:.2f}')

# ──────────────────────────────────────────────────────────────
# STEP 3 — TRAIN MULTIPLE MODELS
# ──────────────────────────────────────────────────────────────

In [ ]:
print('\n[3/6] Training models...')

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced',
                                               max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(class_weight='balanced',
                                                    random_state=42),
    'Random Forest'      : RandomForestClassifier(class_weight='balanced',
                                                    n_estimators=200,
                                                    random_state=42),
    'XGBoost'            : XGBClassifier(scale_pos_weight=scale_pos,
                                           eval_metric='logloss',
                                           use_label_encoder=False,
                                           random_state=42),
}

results = []
trained = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall'   : round(recall_score(y_test, y_pred), 4),
        'F1'       : round(f1_score(y_test, y_pred), 4),
        'ROC-AUC'  : round(roc_auc_score(y_test, y_proba), 4),
    })
    trained[name] = model
    print(f'  {name:<22} trained')

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print('\n=== Model Comparison ===')
print(results_df.to_string(index=False))


# ──────────────────────────────────────────────────────────────
# STEP 4 — PICK BEST MODEL
# ──────────────────────────────────────────────────────────────

In [ ]:
best_name  = results_df.iloc[0]['Model']
best_model = trained[best_name]
print(f'\n[4/6] Best model: {best_name}')

# ──────────────────────────────────────────────────────────────
# STEP 5 — DETAILED EVALUATION OF BEST MODEL
# ──────────────────────────────────────────────────────────────

In [ ]:
print(f'\n[5/6] Detailed evaluation — {best_name}')
y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print('\nConfusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
print(f'                 Pred:Existing  Pred:Attrited')
print(f'  Actual:Existing {cm[0,0]:>8}       {cm[0,1]:>8}')
print(f'  Actual:Attrited {cm[1,0]:>8}       {cm[1,1]:>8}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred,
                             target_names=['Existing', 'Attrited']))

# Feature importance
if hasattr(best_model, 'feature_importances_'):
    imp = pd.DataFrame({
        'Feature'   : feature_names,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    print('\nTop 10 Important Features:')
    print(imp.head(10).to_string(index=False))

# ──────────────────────────────────────────────────────────────
# STEP 6 — SAVE MODEL + ENCODERS
# ──────────────────────────────────────────────────────────────

In [ ]:
print(f'\n[6/6] Saving model artifacts...')

artifacts = {
    'model'        : best_model,
    'encoders'     : encoders,
    'feature_names': feature_names,
    'model_name'   : best_name,
    'cap_values'   : {col: {'Q1': float(df[col].quantile(0.25)),
                             'Q3': float(df[col].quantile(0.75))}
                       for col in cap_cols},
}

joblib.dump(artifacts, rf'{MODEL_DIR}\churn_model.pkl')
print(f'  Saved: churn_model.pkl')

# Save report
report_path = rf'{MODEL_DIR}\model_report.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('CHURN PREDICTION — MODEL REPORT\n')
    f.write('=' * 60 + '\n\n')
    f.write('Model Comparison:\n')
    f.write(results_df.to_string(index=False) + '\n\n')
    f.write(f'Best Model: {best_name}\n\n')
    f.write('Classification Report:\n')
    f.write(classification_report(y_test, y_pred,
                                   target_names=['Existing', 'Attrited']))
    if hasattr(best_model, 'feature_importances_'):
        f.write('\nFeature Importance:\n')
        f.write(imp.to_string(index=False))

print(f'  Saved: model_report.txt')
print('\n' + '=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)